# Original Data Rebalance Continue-Training

This notebook tests a narrow data hypothesis for the NVIDIA Nemotron reasoning competition: equation tasks may be weak because they are underrepresented in `train.csv`.

It uses only original labeled rows, with no synthetic augmentation:

- all original rows once,
- `equation_numeric` rows repeated 3 extra times,
- `equation_symbolic` rows repeated 2 extra times.

The notebook continues from the Kien `0.86` adapter with a very small learning rate and assistant-only loss masking, then writes `/kaggle/working/submission.zip`.


In [1]:
# Optional Kaggle dependency manager commands.
# Keep these commented so Run All does not reinstall packages.

# Most Kaggle images include core HF packages, but Nemotron training also needs Mamba kernels.
# If mamba_ssm or causal_conv1d is missing, attach a compatible offline wheelhouse.
# pip install --no-index --find-links=/kaggle/input/<your-wheelhouse> mamba-ssm causal-conv1d


In [2]:
from pathlib import Path
from collections import Counter
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import zipfile

import pandas as pd

RANDOM_SEED = 2026
EQUATION_NUMERIC_EXTRA_COPIES = 3
EQUATION_SYMBOLIC_EXTRA_COPIES = 2

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 20
LEARNING_RATE = 2e-6
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
WARMUP_RATIO = 0.03

KAGGLE_INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = WORKING / 'original_data_rebalance_train'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REBALANCED_JSONL = OUTPUT_DIR / 'rebalanced_train.jsonl'
TRAINED_ADAPTER_DIR = OUTPUT_DIR / 'adapter'
ADAPTER_OUTPUT_DIR = TRAINED_ADAPTER_DIR
MANIFEST_PATH = OUTPUT_DIR / 'manifest.json'
SUBMISSION_ZIP = WORKING / 'submission.zip'
BASE_MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

MODEL_PATH_CANDIDATES = [
    Path('/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'),
    Path('/kaggle/input/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'),
    Path('/kaggle/input/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'),
    Path('/kaggle/input/nemotron-3-nano-30b-a3b-bf16'),
]

ADAPTER_PATH_CANDIDATES = [
    Path('/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1'),
    Path('/kaggle/input/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1'),
    Path('/kaggle/input/nemotron-nano-30b-trained/triton/tinker-adapter/1'),
    Path('/kaggle/input/tinker-adapter/1'),
]

TRAIN_CSV_CANDIDATES = [
    Path('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'),
    Path('/kaggle/input/nvidia-nemotron-model-reasoning-challenge/train.csv'),
    Path('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv'),
    Path('data/train.csv'),
]

print('Working:', WORKING)
print('Output:', OUTPUT_DIR)


Working: /kaggle/working
Output: /kaggle/working/original_data_rebalance_train


In [3]:
# Optional offline install for Nemotron Mamba kernel dependencies.
# Supports both wheelhouse folders and explicit .whl file paths.

INSTALL_OFFLINE_KERNEL_DEPS = True

KERNEL_WHEELHOUSE_CANDIDATES = [
    Path('/kaggle/input/datasets/grooking/mamba-ssm'),
    Path('/kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels'),
    Path('/kaggle/input/datasets/chakrabhuaanavdeva/mamba-ssm/wheels'),
    Path('/kaggle/input/datasets/chakrabhuaanavdeva/mamba-ssn/wheels'),
]
CAUSAL_CONV1D_WHEEL_CANDIDATES = [
    Path('/kaggle/input/datasets/grooking/mamba-ssm/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl'),
    Path('/kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/causal_conv1d-1.6.0-cp312-cp312-linux_x86_64.whl'),
]
MAMBA_SSM_WHEEL_CANDIDATES = [
    Path('/kaggle/input/datasets/grooking/mamba-ssm/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl'),
    Path('/kaggle/input/datasets/chakrabhuanavdeva/mamba-ssn/wheels/mamba_ssm-2.3.0-cp312-cp312-linux_x86_64.whl'),
]
CAUSAL_CONV1D_SOURCE = Path('/kaggle/input/datasets/yashbishnoi98/casual-convd1/causal_conv1d-1.6.1')


def find_existing_file(candidates):
    return next((p for p in candidates if p.exists() and p.is_file()), None)


def find_existing_dir(candidates):
    return next((p for p in candidates if p.exists() and p.is_dir()), None)


KERNEL_WHEELHOUSE = find_existing_dir(KERNEL_WHEELHOUSE_CANDIDATES)
CAUSAL_CONV1D_WHEEL = find_existing_file(CAUSAL_CONV1D_WHEEL_CANDIDATES)
MAMBA_SSM_WHEEL = find_existing_file(MAMBA_SSM_WHEEL_CANDIDATES)


def install_missing_package(import_name, pip_args):
    if importlib.util.find_spec(import_name) is not None:
        print(f'{import_name}: already available')
        return True
    if not INSTALL_OFFLINE_KERNEL_DEPS:
        print(f'{import_name}: missing; offline install disabled')
        return False
    import subprocess
    import sys
    print(f'{import_name}: installing with', pip_args)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pip_args])
    return importlib.util.find_spec(import_name) is not None


if INSTALL_OFFLINE_KERNEL_DEPS:
    print('KERNEL_WHEELHOUSE:', KERNEL_WHEELHOUSE)
    print('CAUSAL_CONV1D_WHEEL:', CAUSAL_CONV1D_WHEEL)
    print('MAMBA_SSM_WHEEL:', MAMBA_SSM_WHEEL)
    if CAUSAL_CONV1D_WHEEL is not None:
        install_missing_package('causal_conv1d', [str(CAUSAL_CONV1D_WHEEL)])
    elif KERNEL_WHEELHOUSE is not None:
        install_missing_package('causal_conv1d', ['--no-index', f'--find-links={KERNEL_WHEELHOUSE}', 'causal-conv1d'])
    elif CAUSAL_CONV1D_SOURCE.exists():
        install_missing_package('causal_conv1d', ['--no-build-isolation', str(CAUSAL_CONV1D_SOURCE)])
    else:
        print('causal_conv1d wheel/source not found')

    if MAMBA_SSM_WHEEL is not None:
        install_missing_package('mamba_ssm', [str(MAMBA_SSM_WHEEL)])
    elif KERNEL_WHEELHOUSE is not None:
        install_missing_package('mamba_ssm', ['--no-index', f'--find-links={KERNEL_WHEELHOUSE}', 'mamba-ssm'])
    else:
        print('mamba_ssm wheelhouse/wheel not found')


KERNEL_WHEELHOUSE: /kaggle/input/datasets/grooking/mamba-ssm
CAUSAL_CONV1D_WHEEL: /kaggle/input/datasets/grooking/mamba-ssm/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl
MAMBA_SSM_WHEEL: /kaggle/input/datasets/grooking/mamba-ssm/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl
causal_conv1d: installing with ['/kaggle/input/datasets/grooking/mamba-ssm/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl']
Processing /kaggle/input/datasets/grooking/mamba-ssm/causal_conv1d-1.6.1-cp312-cp312-linux_x86_64.whl
mamba_ssm: installing with ['/kaggle/input/datasets/grooking/mamba-ssm/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl']
Processing /kaggle/input/datasets/grooking/mamba-ssm/mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl


In [4]:
def find_existing_path(candidates, required_files=None, recursive_name=None):
    required_files = required_files or []
    for path in candidates:
        if path.exists() and all((path / name).exists() for name in required_files):
            return path
    if recursive_name and KAGGLE_INPUT.exists():
        for hit in KAGGLE_INPUT.rglob(recursive_name):
            root = hit.parent
            if all((root / name).exists() for name in required_files):
                return root
    return None


def find_train_csv():
    for path in TRAIN_CSV_CANDIDATES:
        if path.exists():
            return path
    if KAGGLE_INPUT.exists():
        hits = sorted(KAGGLE_INPUT.rglob('train.csv'))
        if hits:
            return hits[0]
    return None


def find_base_model_path():
    explicit = os.environ.get('BASE_MODEL_PATH')
    if explicit and Path(explicit).exists() and (Path(explicit) / 'config.json').exists():
        return Path(explicit)
    path = find_existing_path(MODEL_PATH_CANDIDATES, required_files=['config.json'])
    if path is not None:
        return path
    if KAGGLE_INPUT.exists():
        hits = []
        for cfg in KAGGLE_INPUT.rglob('config.json'):
            root = cfg.parent
            low = str(root).lower()
            if 'nemotron' in low and 'adapter' not in low:
                hits.append(root)
        if hits:
            hits.sort(key=lambda p: ('30b' not in str(p).lower(), 'nano' not in str(p).lower(), len(str(p))))
            return hits[0]
    return None


def find_kien_adapter_path():
    explicit = os.environ.get('KIEN_ADAPTER_PATH')
    if explicit and Path(explicit).exists():
        p = Path(explicit)
        if (p / 'adapter_config.json').exists() and (p / 'adapter_model.safetensors').exists():
            return p
    return find_existing_path(
        ADAPTER_PATH_CANDIDATES,
        required_files=['adapter_config.json', 'adapter_model.safetensors'],
        recursive_name='adapter_config.json',
    )


TRAIN_CSV_PATH = find_train_csv()
BASE_MODEL_PATH = find_base_model_path()
KIEN_ADAPTER_PATH = find_kien_adapter_path()

if TRAIN_CSV_PATH is None:
    raise FileNotFoundError('train.csv not found. Attach competition data input.')
if BASE_MODEL_PATH is None:
    raise FileNotFoundError('Nemotron base model not found. Attach base model input or set BASE_MODEL_PATH.')
if KIEN_ADAPTER_PATH is None:
    raise FileNotFoundError('Kien adapter not found. Attach Kien Tinker adapter v1 or set KIEN_ADAPTER_PATH.')

TRAIN_CSV = TRAIN_CSV_PATH
BASE_ADAPTER_PATH = KIEN_ADAPTER_PATH

print('Train CSV:', TRAIN_CSV_PATH)
print('Base model:', BASE_MODEL_PATH)
print('Kien adapter:', KIEN_ADAPTER_PATH)


Train CSV: /kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv
Base model: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Kien adapter: /kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1


In [5]:
FINAL_ANSWER_INSTRUCTION = (
    'Reason step-by-step, but keep the reasoning concise and efficient.\n'
    'After completing your reasoning, output exactly one final answer inside a single LaTeX \\boxed{} command.\n'
    'The box must contain only the final answer, and no text may appear after the final \\boxed{}.'
)


def infer_task_family(prompt):
    text = str(prompt)
    lower = text.lower()
    if 'alice' in lower and 'wonderland' in lower and 'equations' in lower:
        symbols = set("!@#$%^&*()_+-=[]{}|;:'\",.<>/?`~\\")
        if any(ch in symbols for ch in text.split('now, determine the result for:')[-1].strip()):
            return 'equation_symbolic'
        return 'equation_numeric'
    if 'roman numeral' in lower or 'roman' in lower:
        return 'numeral'
    if 'cipher' in lower or 'encrypted' in lower or 'decrypt' in lower:
        return 'cipher'
    if 'gravity' in lower or 'planet' in lower or 'mass' in lower:
        return 'gravity'
    if 'convert' in lower and ('unit' in lower or 'meter' in lower or 'gram' in lower or 'liter' in lower):
        return 'unit_conversion'
    if 'bit' in lower or 'binary' in lower:
        return 'bit_manipulation'
    return 'other'


def format_record(row, task_family, source_tag):
    prompt = str(row['prompt'])
    answer = str(row['answer'])
    user = prompt + '\n' + FINAL_ANSWER_INSTRUCTION
    assistant = f'Apply the demonstrated rule carefully.\n\\boxed{{{answer}}}'
    return {
        'id': str(row.get('id', '')),
        'task': task_family,
        'source': source_tag,
        'prompt': prompt,
        'answer': answer,
        'text': f'<|user|>\n{user}\n<|assistant|>\n{assistant}',
    }


def build_rebalanced_records(df, seed=RANDOM_SEED):
    work = df.copy()
    work['task'] = work['prompt'].map(infer_task_family)
    source_counts_before = work['task'].value_counts().sort_index().to_dict()

    pieces = [work.assign(source='original')]
    numeric = work[work['task'] == 'equation_numeric']
    symbolic = work[work['task'] == 'equation_symbolic']

    for idx in range(EQUATION_NUMERIC_EXTRA_COPIES):
        pieces.append(numeric.assign(source=f'equation_numeric_extra_{idx + 1}'))
    for idx in range(EQUATION_SYMBOLIC_EXTRA_COPIES):
        pieces.append(symbolic.assign(source=f'equation_symbolic_extra_{idx + 1}'))

    balanced = pd.concat(pieces, ignore_index=True)
    balanced = balanced.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    source_counts_after = balanced['task'].value_counts().sort_index().to_dict()
    records = [format_record(row, row['task'], row['source']) for _, row in balanced.iterrows()]
    return records, source_counts_before, source_counts_after


train_df = pd.read_csv(TRAIN_CSV_PATH)
records, source_counts_before, source_counts_after = build_rebalanced_records(train_df)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with REBALANCED_JSONL.open('w', encoding='utf-8') as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print('Original rows:', len(train_df))
print('Rebalanced rows:', len(records))
print('source_counts_before:', source_counts_before)
print('source_counts_after:', source_counts_after)
print('Wrote:', REBALANCED_JSONL)


Original rows: 9500
Rebalanced rows: 12610
source_counts_before: {'bit_manipulation': 1602, 'cipher': 1576, 'equation_symbolic': 1555, 'other': 3173, 'unit_conversion': 1594}
source_counts_after: {'bit_manipulation': 1602, 'cipher': 1576, 'equation_symbolic': 4665, 'other': 3173, 'unit_conversion': 1594}
Wrote: /kaggle/working/original_data_rebalance_train/rebalanced_train.jsonl


In [6]:
def ensure_training_dependencies():
    missing = [
        pkg for pkg in ['torch', 'transformers', 'peft', 'datasets', 'accelerate', 'safetensors', 'mamba_ssm', 'causal_conv1d']
        if importlib.util.find_spec(pkg) is None
    ]
    if missing:
        raise ImportError(
            'Training dependencies are missing: ' + ', '.join(missing) + '. '
            'Attach/install a compatible offline wheelhouse before running this notebook. '
            'Stopping before packaging so the unchanged baseline is not submitted by accident.'
        )


ensure_training_dependencies()

import torch
from datasets import load_dataset
from peft import PeftModel
from safetensors.torch import load_file, save_file
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is required for continue-training this adapter.')


def prepare_adapter_for_training(adapter_path):
    adapter_path = Path(adapter_path)
    source_model = adapter_path / 'adapter_model.safetensors'
    source_config = adapter_path / 'adapter_config.json'
    if not source_model.exists() or not source_config.exists():
        raise FileNotFoundError(f'Adapter files not found in {adapter_path}')

    prepared_dir = OUTPUT_DIR / 'kien_adapter_for_training'
    if prepared_dir.exists():
        shutil.rmtree(prepared_dir)
    prepared_dir.mkdir(parents=True, exist_ok=True)

    with source_config.open('r', encoding='utf-8') as f:
        config = json.load(f)
    config['base_model_name_or_path'] = str(BASE_MODEL_PATH)

    tensors = load_file(str(source_model))
    normalized = {}
    rename_count = 0
    for key, tensor in tensors.items():
        new_key = key.replace('base_model.model.model.layers.', 'base_model.model.backbone.layers.')
        if new_key != key:
            rename_count += 1
        if new_key in normalized:
            raise ValueError(f'Adapter key collision after namespace normalization: {new_key}')
        normalized[new_key] = tensor

    with (prepared_dir / 'adapter_config.json').open('w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, sort_keys=True)
    save_file(normalized, str(prepared_dir / 'adapter_model.safetensors'))

    sample_keys = list(normalized)[:5]
    print('Prepared training adapter:', prepared_dir)
    print('Adapter tensors:', len(normalized), 'renamed:', rename_count)
    print('Prepared key sample:', sample_keys)
    return prepared_dir


TRAINING_ADAPTER_PATH = prepare_adapter_for_training(KIEN_ADAPTER_PATH)


def load_training_model_and_tokenizer():
    tokenizer = AutoTokenizer.from_pretrained(str(BASE_MODEL_PATH), trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        str(BASE_MODEL_PATH),
        trust_remote_code=True,
        dtype=torch.bfloat16,
        device_map='auto',
    )
    model = PeftModel.from_pretrained(model, str(TRAINING_ADAPTER_PATH), is_trainable=True)
    model.print_trainable_parameters()
    return model, tokenizer


model, tokenizer = load_training_model_and_tokenizer()


torch: 2.10.0+cu128
CUDA available: True
Prepared training adapter: /kaggle/working/original_data_rebalance_train/kien_adapter_for_training
Adapter tensors: 12010 renamed: 12008
Prepared key sample: ['base_model.model.lm_head.lora_A.weight', 'base_model.model.lm_head.lora_B.weight', 'base_model.model.backbone.layers.0.mixer.in_proj.lora_A.weight', 'base_model.model.backbone.layers.0.mixer.in_proj.lora_B.weight', 'base_model.model.backbone.layers.0.mixer.out_proj.lora_A.weight']


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

trainable params: 888,154,112 || all params: 32,466,091,456 || trainable%: 2.7356


In [7]:
raw_dataset = load_dataset('json', data_files=str(REBALANCED_JSONL), split='train')


def tokenize_batch(batch):
    tokenized = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )
    assistant_marker = '<|assistant|>\n'
    labels = []
    for text, input_ids in zip(batch['text'], tokenized['input_ids']):
        marker_index = text.rfind(assistant_marker)
        if marker_index < 0:
            raise ValueError('assistant marker not found in training text')
        prefix = text[: marker_index + len(assistant_marker)]
        prefix_ids = tokenizer(prefix, add_special_tokens=False)['input_ids']
        label = list(input_ids)
        cutoff = min(len(prefix_ids), len(label))
        label[:cutoff] = [-100] * cutoff
        labels.append(label)
    tokenized['labels'] = labels
    return tokenized


tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=raw_dataset.column_names,
    desc='Tokenizing with assistant-only labels',
)


class CompletionOnlyDataCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        input_features = [{'input_ids': f['input_ids'], 'attention_mask': f['attention_mask']} for f in features]
        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        max_len = batch['input_ids'].shape[1]
        padded_labels = []
        for feature in features:
            label = list(feature['labels'])
            label += [-100] * (max_len - len(label))
            padded_labels.append(label)
        batch['labels'] = torch.tensor(padded_labels, dtype=torch.long)
        return batch


training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'trainer'),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    logging_steps=1,
    save_strategy='no',
    report_to=[],
    bf16=True,
    optim='adamw_torch',
    gradient_checkpointing=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=CompletionOnlyDataCollator(tokenizer),
)

trainer.train()
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUTPUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR / 'tokenizer_snapshot'))
print('Saved adapter:', ADAPTER_OUTPUT_DIR)


Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing with assistant-only labels:   0%|          | 0/12610 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
1,49.291187
2,49.705971
3,43.111404
4,43.928074
5,44.462254
6,45.061821
7,44.107716
8,43.409000
9,43.800884
10,43.838619


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Saved adapter: /kaggle/working/original_data_rebalance_train/adapter


In [8]:
def sha256_path(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def normalize_adapter_for_metric(adapter_dir, output_zip):
    adapter_dir = Path(adapter_dir)
    config_path = adapter_dir / 'adapter_config.json'
    model_path = adapter_dir / 'adapter_model.safetensors'
    if not config_path.exists():
        raise FileNotFoundError(config_path)
    if not model_path.exists():
        raise FileNotFoundError(model_path)

    with config_path.open('r', encoding='utf-8') as f:
        config = json.load(f)
    config['base_model_name_or_path'] = BASE_MODEL_NAME

    tmp_dir = OUTPUT_DIR / 'metric_adapter'
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    with (tmp_dir / 'adapter_config.json').open('w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, sort_keys=True)
    shutil.copy2(model_path, tmp_dir / 'adapter_model.safetensors')

    if output_zip.exists():
        output_zip.unlink()
    with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
        zf.write(tmp_dir / 'adapter_config.json', arcname='adapter_config.json')
        zf.write(tmp_dir / 'adapter_model.safetensors', arcname='adapter_model.safetensors')
    default_submission = Path('/kaggle/working/submission.zip')
    if output_zip.resolve() != default_submission.resolve():
        shutil.copy2(output_zip, default_submission)
    return tmp_dir


metric_adapter_dir = normalize_adapter_for_metric(ADAPTER_OUTPUT_DIR, SUBMISSION_ZIP)
manifest = {
    'method': 'original_data_rebalance_train',
    'base_model': str(BASE_MODEL_PATH),
    'base_model_name_for_metric': BASE_MODEL_NAME,
    'source_adapter': str(BASE_ADAPTER_PATH),
    'train_csv': str(TRAIN_CSV_PATH),
    'rebalanced_jsonl': str(REBALANCED_JSONL),
    'source_counts_before': source_counts_before,
    'source_counts_after': source_counts_after,
    'settings': {
        'equation_numeric_extra_copies': EQUATION_NUMERIC_EXTRA_COPIES,
        'equation_symbolic_extra_copies': EQUATION_SYMBOLIC_EXTRA_COPIES,
        'max_steps': MAX_STEPS,
        'learning_rate': LEARNING_RATE,
        'max_seq_length': MAX_SEQ_LENGTH,
        'per_device_train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    },
    'output': {
        'adapter_dir': str(ADAPTER_OUTPUT_DIR),
        'metric_adapter_dir': str(metric_adapter_dir),
        'submission_zip': str(SUBMISSION_ZIP),
        'default_submission_zip': '/kaggle/working/submission.zip',
        'submission_zip_sha256': sha256_path(SUBMISSION_ZIP),
        'zip_contents': sorted(zipfile.ZipFile(SUBMISSION_ZIP).namelist()),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(json.dumps(manifest, indent=2, sort_keys=True))

{
  "base_model": "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
  "base_model_name_for_metric": "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16",
  "method": "original_data_rebalance_train",
  "output": {
    "adapter_dir": "/kaggle/working/original_data_rebalance_train/adapter",
    "default_submission_zip": "/kaggle/working/submission.zip",
    "metric_adapter_dir": "/kaggle/working/original_data_rebalance_train/metric_adapter",
    "submission_zip": "/kaggle/working/submission.zip",
    "submission_zip_sha256": "7711a9c0643decd1283445b2f809613b20af23eb6aef15f6f2e06c82e6fe4f0f",
    "zip_contents": [
      "adapter_config.json",
      "adapter_model.safetensors"
    ]
  },
  "rebalanced_jsonl": "/kaggle/working/original_data_rebalance_train/rebalanced_train.jsonl",
  "settings": {
    "equation_numeric_extra_copies": 3,
    "equation_symbolic_extra_copies": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-06,
    "max_seq_length": 2048